In [1]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import input_file_name, col, count
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
# 1. Hardware/Environment Overrides
os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/openjdk-17.jdk/Contents/Home"
os.environ["PYSPARK_PYTHON"] = sys.executable

# 2. Hardened Spark 4.1.1 Session
spark = SparkSession.builder \
    .appName("BronzeValidation") \
    .config("spark.jars.packages", (
        "org.apache.hadoop:hadoop-azure:3.4.1,"
        "com.microsoft.azure:azure-storage:8.6.6,"
        "io.delta:delta-spark_2.13:4.1.0"
    )) \
    .config("spark.hadoop.fs.azure.account.auth.type", "SharedKey") \
    .config("spark.hadoop.fs.abfss.impl", "org.apache.hadoop.fs.azurebfs.SecureAzureBlobFileSystem") \
    .getOrCreate()

# 3. Credentials
storage_account_name = "coinmarketcapapi"
container_name = "coin-market-cap-api-data"
storage_account_key = os.getenv("AZURE_STORAGE_KEY")

spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)

# 4. Path Definition
bronze_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/bronze"

print(f"✅ Connected to Bronze Layer at: {bronze_path}")

✅ Connected to Bronze Layer at: abfss://coin-market-cap-api-data@coinmarketcapapi.dfs.core.windows.net/bronze


In [6]:
# Load all raw JSON files
# Using recursiveFileLookup to find files in date-partitioned folders
raw_df = spark.read.option("recursiveFileLookup", "true").json(bronze_path)

# Add the file path so we can see which folder/file the data came from
raw_df_with_source = raw_df.withColumn("source_file", input_file_name())

print(f"📊 Total Raw Records Found: {raw_df_with_source.count()}")
raw_df_with_source.select("source", "endpoint_type", "source_file").show(10, truncate=False)

📊 Total Raw Records Found: 5


+-------------+-------------------+-------------------------------------------------------------------------------------------------------------------------------------------------+
|source       |endpoint_type      |source_file                                                                                                                                      |
+-------------+-------------------+-------------------------------------------------------------------------------------------------------------------------------------------------+
|coinmarketcap|listings           |abfss://coin-market-cap-api-data@coinmarketcapapi.dfs.core.windows.net/bronze/coinmarketcap/ingestion_date=2026-04-16/listings_121005.json       |
|coingecko    |historical_backfill|abfss://coin-market-cap-api-data@coinmarketcapapi.dfs.core.windows.net/bronze/coingecko/ingestion_date=2026-04-16/historical_backfill_121006.json|
|coingecko    |markets_snapshot   |abfss://coin-market-cap-api-data@coinmarketcapapi.dfs.c

In [7]:
print("🔍 Inspecting Raw JSON Structure:")
raw_df_with_source.printSchema()

# Check for specific endpoints found in Bronze
raw_df_with_source.groupBy("source", "endpoint_type").count().show()

🔍 Inspecting Raw JSON Structure:
root
 |-- coin_id: string (nullable = true)
 |-- endpoint_type: string (nullable = true)
 |-- ingested_at: string (nullable = true)
 |-- payload: string (nullable = true)
 |-- source: string (nullable = true)
 |-- source_file: string (nullable = false)



+-------------+-------------------+-----+
|       source|      endpoint_type|count|
+-------------+-------------------+-----+
|coinmarketcap|           listings|    1|
|    coingecko|historical_backfill|    1|
|    coingecko|   markets_snapshot|    1|
|coinmarketcap|             quotes|    1|
|coinmarketcap|             global|    1|
+-------------+-------------------+-----+



In [8]:
# Check if the payload is arriving empty
null_payload_count = raw_df_with_source.filter(col("payload").isNull()).count()

if null_payload_count > 0:
    print(f"⚠️ WARNING: Found {null_payload_count} records with empty payloads!")
else:
    print("✅ Success: All payloads contain data.")

# Sample view of the nested payload
print("👀 Sample Payload Content:")
raw_df_with_source.select("endpoint_type", "payload").limit(3).show(truncate=80)

✅ Success: All payloads contain data.
👀 Sample Payload Content:


+-------------------+--------------------------------------------------------------------------------+
|      endpoint_type|                                                                         payload|
+-------------------+--------------------------------------------------------------------------------+
|           listings|[{"id": 1, "name": "Bitcoin", "symbol": "BTC", "slug": "bitcoin", "infinite_s...|
|historical_backfill|{"prices": [[1773749044863, 2327.5732272081914], [1773752492336, 2315.8584872...|
|   markets_snapshot|[{"id": "bitcoin", "symbol": "btc", "name": "Bitcoin", "image": "https://coin...|
+-------------------+--------------------------------------------------------------------------------+



In [9]:
spark.stop()